In [1]:
%run common_imports.py

%matplotlib qt
%config InlineBackend.figure_format = 'retina'
sns.set_context("talk")

%reload_ext autoreload
%autoreload 2
pd.options.display.max_rows = 600
pd.set_option('display.float_format', lambda x: '%.9f' % x)

dj.config['display.limit'] = 10**3  

os.environ["SPYGLASS_USE_TRANSACTIONS"] = "1"  
os.environ['KACHERY_API_KEY'] = "RhysjLwgmBAt2ObCyXXaDnqAv2kTdYRa"

[2026-03-10 18:09:38,739][INFO]: DataJoint is configured from /media/labuser/NA_1_2025/spyglass/wilbur/dj_local_conf.json
[2026-03-10 18:09:39,221][INFO]: DataJoint 0.14.9 connected to anirudh@172.16.102.154:3306


In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats                                                                                 

### Load data

In [3]:
#extract position data
#read from csv
trialized_position = pd.read_csv("/media/labuser/NA_1_2025/spyglass/wilbur/analysis/position/trialized_position.csv", index_col = "time")

In [4]:
#extract spikes
#read from npz
data = np.load("/media/labuser/NA_1_2025/spyglass/wilbur/analysis/final_spikes/mfpc_spikes.npz", allow_pickle=True)
mpfc_spikes = [data[f"arr_{i}"] for i in range(len(data.files))]


### Prepare dataframe for regression

In [42]:
def fit_glm_all_units(formula: str,
                      cov_df: pd.DataFrame,
                      spike_counts_masked: np.array,
                      unit_ids: np.array,
                      bin_size = 0.002):
    
    rows = []
    for i, uid in enumerate(unit_ids):
        df = cov_df.copy()
        df["spike_count"] = spike_counts_masked[i]  # pre-masked counts
        try:
            res = smf.glm(formula, data=df, family=sm.families.Poisson()).fit(disp=False)
            rows.append(dict(                                                                                       
                unit=uid,   
                aic=res.aic,
                llf=res.llf,
                deviance=res.deviance,
                n_params=len(res.params),
                n_obs=int(res.nobs),
                converged=res.converged,
                coef=res.params.to_dict(),
                bse=res.bse.to_dict(),
                deviance_null = res.null_deviance,
                df_model = res.df_model
            ))

        except Exception as e:
            rows.append(dict(
                unit=uid, aic=np.nan, llf=np.nan, deviance=np.nan,
                n_params=np.nan, n_obs=np.nan, converged=False,
                coef=None, bse=None, deviance_null=np.nan,
                df_model=np.nan, error=str(e)          # ← add error
            ))


    return pd.DataFrame(rows)

In [6]:
BIN_SIZE = 0.002  

bin_edges = np.arange(
    trialized_position.index.min(), trialized_position.index.max() + BIN_SIZE, BIN_SIZE
)
bin_centers = bin_edges[:-1] + BIN_SIZE / 2

spike_counts = np.array([np.histogram(spikes, bins=bin_edges)[0] for spikes in mpfc_spikes]) 

In [7]:
def interp_col(col_values, times, bin_centers):
    if pd.api.types.is_numeric_dtype(col_values):
        return np.interp(bin_centers, times, col_values.astype(float))
    else:
        idx = np.searchsorted(times, bin_centers).clip(0, len(times) - 1)
        return col_values.iloc[idx].values

cols_to_interp = [c for c in trialized_position.columns if c != "video_frame_ind"]
times = trialized_position.index.astype(float).values

interpolated = {col: interp_col(trialized_position[col], times, bin_centers) for col in cols_to_interp}

interp_trialised_position = pd.DataFrame(interpolated, columns=cols_to_interp)
interp_trialised_position.insert(0, "time_bin_center", bin_centers)

mask = (interp_trialised_position["zone"]=="run") &\
    (interp_trialised_position["trial_type"].isin(["outbound", "inbound"]))


cov_df = interp_trialised_position[mask]
spike_counts_masked = spike_counts[:, mask]
unit_ids = np.arange(0, len(spike_counts_masked))

# print(cov_df.head(1))
# print(spike_counts_masked[0].shape)
#print(unit_ids)

In [54]:
cov_df = cov_df.rename(columns={"left/right": "choice"})

### Models:

#### Single variable models:
1. Null model (constant rate)
2. spike_count ~ trial_type (categorical)
3. spike_count ~ left/right choice (categorical)
4. spike_count ~ speed (linear)
5. spike_count ~ bs(speed, df = ) (spline)
6. spike_coun ~ bs(linear_position, df = ) (spline)

#### Mutli-variable models:

### Null model

#### Fit on one unit 

In [55]:
unit_idx = 9
spk_cov_df = cov_df.copy()
spk_cov_df["spike_count"] = spike_counts_masked[unit_idx]

In [9]:
model_constant = smf.glm("spike_count ~ 1", data=spk_cov_df, family=sm.families.Poisson())
results_constant = model_constant.fit()

print(results_constant.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1835243
Model:                            GLM   Df Residuals:                  1835242
Model Family:                 Poisson   Df Model:                            0
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -15493.
Date:                Tue, 10 Mar 2026   Deviance:                       27031.
Time:                        18:09:56   Pearson chi2:                 1.83e+06
No. Iterations:                     8   Pseudo R-squ. (CS):              0.000
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -6.8328      0.022   -303.889      0.0

In [10]:
# Interpret the coefficient
mean_count_per_bin = np.exp(results_constant.params["Intercept"])
mean_rate_hz = mean_count_per_bin / BIN_SIZE

print(f"β₀ = {results_constant.params['Intercept']:.4f}")
print(f"exp(β₀) = {mean_count_per_bin:.4f} spikes/bin")
print(f"Firing rate = {mean_rate_hz:.2f} Hz")
print(f"Observed mean = {spk_cov_df['spike_count'].mean():.4f} spikes/bin")

β₀ = -6.8328
exp(β₀) = 0.0011 spikes/bin
Firing rate = 0.54 Hz
Observed mean = 0.0011 spikes/bin


#### Fit on all units

In [ ]:
# null_model_all = fit_glm_all_units("spike_count ~ 1", cov_df, spike_counts_masked, unit_ids)

/home/labuser/miniforge3/envs/spyglass/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:445: RuntimeWarning: invalid value encountered in divide
  endog_mu = self._clean(endog / mu)


In [43]:
# null_model_all.to_csv(f"{base_dir}/analysis/null_model_all.csv")
null_model_all = pd.read_csv(f"{base_dir}/analysis/null_model_all.csv", index_col=0)
null_model_all["model"] = "null"

In [44]:
null_model_all.head()

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,model
0,0,146661.528243705,-73329.764121853,122310.026719650,1,1835243,True,{'Intercept': -5.014316233297876},{'Intercept': 0.009057287368239611},null
1,1,62875.359086194,-31436.679543097,53915.517969278,1,1835243,True,{'Intercept': -6.01508594098356},{'Intercept': 0.01493869039855228},null
2,2,27650.746202234,-13824.373101117,24176.746202234,1,1835243,True,{'Intercept': -6.963348560546998},{'Intercept': 0.024000746649013627},null
3,3,83401.915912946,-41699.957956473,70930.847384752,1,1835243,True,{'Intercept': -5.684272558606533},{'Intercept': 0.012661271297716607},null
4,4,115959.900830617,-57978.950415308,97535.900830617,1,1835243,True,{'Intercept': -5.294533754779503},{'Intercept': 0.010419493518636521},null


### Trial type

#### Fit on one unit

In [25]:
model_trial_type = smf.glm("spike_count ~ trial_type", data=spk_cov_df, family=sm.families.Poisson())
results_trial_type = model_trial_type.fit()

print(results_trial_type.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1835243
Model:                            GLM   Df Residuals:                  1835241
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -15273.
Date:                Tue, 10 Mar 2026   Deviance:                       26590.
Time:                        18:25:15   Pearson chi2:                 1.83e+06
No. Iterations:                     9   Pseudo R-squ. (CS):          0.0002400
Covariance Type:            nonrobust                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -6

In [31]:
rate_inbound  = np.exp(results_trial_type.params["Intercept"]) / BIN_SIZE        # Hz
rate_outbound = np.exp(results_trial_type.params["Intercept"] + results_trial_type.params["trial_type[T.outbound]"]) / BIN_SIZE
ratio         = np.exp(results_trial_type.params["trial_type[T.outbound]"])      # outbound/inbound rate ratio

print("inbound rate: ", rate_inbound)
print("outbound rate: ", rate_outbound)
print("outbound/inbound: ", ratio)

inbound rate:  0.7729987805818047
outbound rate:  0.2777832207690415
outbound/inbound:  0.3593579029451577


#### Fit all units 

In [ ]:
# trial_type_model_all = fit_glm_all_units("spike_count ~ trial_type", cov_df, spike_counts_masked, unit_ids)

/home/labuser/miniforge3/envs/spyglass/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:445: RuntimeWarning: invalid value encountered in divide
  endog_mu = self._clean(endog / mu)


In [48]:
# trial_type_model_all.to_csv(f"{base_dir}/analysis/trial_type_model_all.csv")
trial_type_model_all = pd.read_csv(f"{base_dir}/analysis/trial_type_model_all.csv", index_col=0)
trial_type_model_all["model"] = "trial_type"

In [49]:
trial_type_model_all.head()

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,model
0,0,146354.172384433,-73175.086192217,122000.670860378,2,1835243,True,"{'Intercept': -4.874482451548208, 'trial_type[...","{'Intercept': 0.011631052629827653, 'trial_typ...",trial_type
1,1,62822.257539506,-31409.128769753,53860.416422589,2,1835243,True,"{'Intercept': -5.915530564583954, 'trial_type[...","{'Intercept': 0.01957400707490679, 'trial_type...",trial_type
2,2,27628.592982041,-13812.296491021,24152.592982041,2,1835243,True,"{'Intercept': -7.081904955371571, 'trial_type[...","{'Intercept': 0.03507153119208145, 'trial_type...",trial_type
3,3,83311.912599513,-41653.956299756,70838.844071318,2,1835243,True,"{'Intercept': -5.575779636466067, 'trial_type[...","{'Intercept': 0.016515957975578685, 'trial_typ...",trial_type
4,4,115866.653900175,-57931.326950087,97440.653900175,2,1835243,True,"{'Intercept': -5.202843731694705, 'trial_type[...","{'Intercept': 0.01370634840006244, 'trial_type...",trial_type


### Choice

#### Fit on one unit

In [63]:
choice_mask = spk_cov_df["trial_type"]=="outbound"
choice_spk_cov_df = spk_cov_df[choice_mask]
model_choice= smf.glm("spike_count ~ choice", data=choice_spk_cov_df, family=sm.families.Poisson())
results_choice = model_choice.fit()

print(results_choice.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:               867583
Model:                            GLM   Df Residuals:                   867581
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -4007.2
Date:                Tue, 10 Mar 2026   Deviance:                       7050.5
Time:                        19:05:41   Pearson chi2:                 8.67e+05
No. Iterations:                    10   Pseudo R-squ. (CS):          0.0002019
Covariance Type:            nonrobust                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -6.4212      0.076    -

In [67]:
rate_left  = np.exp(results_choice.params["Intercept"]) / BIN_SIZE        # Hz
rate_right = np.exp(results_choice.params["Intercept"] + results_choice.params["choice[T.right]"]) / BIN_SIZE
ratio         = np.exp(results_choice.params["choice[T.right]"])      # outbound/inbound rate ratio

print("left rate: ", rate_left)
print("right rate: ", rate_right)
print("right/left: ", ratio)

left rate:  0.8133174791934886
right rate:  0.203945659961309
right/left:  0.2507577485775272


In [ ]:
#TODO: compare this against a null model that is OUTBOUND only

#### Fit for all units

In [71]:
choice_model_all = fit_glm_all_units("spike_count ~ choice", cov_df[choice_mask], spike_counts_masked[:, choice_mask], unit_ids) #fit only on outbound trials

/home/labuser/miniforge3/envs/spyglass/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:445: RuntimeWarning: invalid value encountered in divide
  endog_mu = self._clean(endog / mu)


In [74]:
choice_model_all.to_csv(f"{base_dir}/analysis/choice_model_all.csv")
choice_model_all = pd.read_csv(f"{base_dir}/analysis/choice_model_all.csv", index_col=0)
choice_model_all["model"] = "choice"

In [75]:
choice_model_all.head()

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,51945.143850145,-25970.571925072,42353.461616311,2.000000000,867583.000000000,True,"{'Intercept': -3.5083827877384812, 'choice[T.r...","{'Intercept': 0.017823067247957607, 'choice[T....",49891.960876831,1.000000000,NaN,choice
1,1,22912.949650976,-11454.474825488,19171.108534059,2.000000000,867583.000000000,True,"{'Intercept': -4.331618392947591, 'choice[T.ri...","{'Intercept': 0.026899608277649346, 'choice[T....",22981.346875516,1.000000000,NaN,choice
2,2,12235.800570111,-6115.900285055,10385.800570111,2.000000000,867583.000000000,True,"{'Intercept': -4.956255211077718, 'choice[T.ri...","{'Intercept': 0.03676073109408769, 'choice[T.r...",12637.415524655,1.000000000,NaN,choice
3,3,32304.974140526,-16150.487070263,27159.746729248,2.000000000,867583.000000000,True,"{'Intercept': -4.2988752543783235, 'choice[T.r...","{'Intercept': 0.026462806167679386, 'choice[T....",29948.910010403,1.000000000,NaN,choice
4,4,40408.210686358,-20202.105343179,32628.210686358,2.000000000,867583.000000000,True,"{'Intercept': -3.5211706857859935, 'choice[T.r...","{'Intercept': 0.017937400083207797, 'choice[T....",42051.179532311,1.000000000,NaN,choice


### Speed

#### Fit for one unit

In [118]:
speed_mask = (cov_df["speed"].notna()) & (cov_df["speed"]>5) & (cov_df["speed"]<120)
cov_df_speed = cov_df[speed_mask].copy()
spike_counts_speed = spike_counts_masked[:, speed_mask]


model_speed = smf.glm("spike_count ~ speed", data= spk_cov_df[(spk_cov_df["speed"].notna()) & (spk_cov_df["speed"] > 5) & (spk_cov_df["speed"] < 120)], family=sm.families.Poisson())
results_speed = model_speed.fit()

print(results_speed.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1132036
Model:                            GLM   Df Residuals:                  1132034
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -11060.
Date:                Wed, 11 Mar 2026   Deviance:                       18861.
Time:                        11:22:42   Pearson chi2:                 1.10e+06
No. Iterations:                    10   Pseudo R-squ. (CS):           0.002169
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -8.3013      0.060   -139.138      0.0

In [119]:
#interpret coefficients
beta0_speed = results_speed.params["Intercept"]
beta1_speed = results_speed.params["speed"]

print("Model interpretation:")
print(f"  β₀ = {beta0_speed:.4f} (log rate at speed=0)")
print(f"  β₁ = {beta1_speed:.5f} (change in log rate per cm/s)")
print()
print(f"At speed=0: rate = {np.exp(beta0_speed) / BIN_SIZE:.2f} Hz")
print(f"At speed=20: rate = {np.exp(beta0_speed + beta1_speed * 20) / BIN_SIZE:.2f} Hz")
print(f"Effect: {100 * (np.exp(beta1_speed) - 1):.2f}% change per 1 cm/s")

Model interpretation:
  β₀ = -8.3013 (log rate at speed=0)
  β₁ = 0.03457 (change in log rate per cm/s)

At speed=0: rate = 0.12 Hz
At speed=20: rate = 0.25 Hz
Effect: 3.52% change per 1 cm/s


#### Fit for all units

In [ ]:
speed_model_all = fit_glm_all_units("spike_count ~ speed", cov_df_speed, spike_counts_speed, unit_ids) 

/home/labuser/miniforge3/envs/spyglass/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:445: RuntimeWarning: invalid value encountered in divide
  endog_mu = self._clean(endog / mu)


In [121]:
speed_model_all["model"] = "speed"
speed_model_all.to_csv(f"{base_dir}/analysis/speed_model_all.csv")
speed_model_all = pd.read_csv(f"{base_dir}/analysis/speed_model_all.csv", index_col=0)

In [122]:
speed_model_all.head()

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,115451.235134141,-57723.617567070,94022.961021363,2.000000000,1132036.000000000,True,"{'Intercept': -5.531505525392574, 'speed': 0.0...","{'Intercept': 0.01741446751010367, 'speed': 0....",100002.515597235,1.000000000,NaN,speed
1,1,48057.687029670,-24026.843514835,40498.459618393,2.000000000,1132036.000000000,True,"{'Intercept': -6.701143112210031, 'speed': 0.0...","{'Intercept': 0.03056836876635762, 'speed': 0....",43103.631638410,1.000000000,NaN,speed
2,2,23998.251519899,-11997.125759949,20670.251519899,2.000000000,1132036.000000000,True,"{'Intercept': -7.447585899666322, 'speed': 0.0...","{'Intercept': 0.04498287310584487, 'speed': 0....",21684.949533751,1.000000000,NaN,speed
3,3,70025.603748604,-35010.801874302,58320.535220409,2.000000000,1132036.000000000,True,"{'Intercept': -6.148447174113083, 'speed': 0.0...","{'Intercept': 0.023659348221290576, 'speed': 0...",61652.360230251,1.000000000,NaN,speed
4,4,95259.836254807,-47627.918127404,77995.836254807,2.000000000,1132036.000000000,True,"{'Intercept': -5.8997085845259045, 'speed': 0....","{'Intercept': 0.02038951377984152, 'speed': 0....",84168.882864172,1.000000000,NaN,speed


### Speed (spline)

#### FIt on one unit


In [130]:
from patsy import bs
from patsy import cr
cr_df = 5
model_speed_spline = smf.glm(f"spike_count ~ cr(speed, df = {cr_df})", data=spk_cov_df[spk_cov_df["speed"].notna()  ], family = sm.families.Poisson())
results_speed_spline = model_speed_spline.fit()
print(results_speed_spline.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1168163
Model:                            GLM   Df Residuals:                  1168158
Model Family:                 Poisson   Df Model:                            4
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -13054.
Date:                Wed, 11 Mar 2026   Deviance:                       22151.
Time:                        15:45:35   Pearson chi2:                 9.59e+05
No. Iterations:                   100   Pseudo R-squ. (CS):           0.002644
Covariance Type:            nonrobust                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept            2.17e+11   2.93

In [131]:
# Predict at specific speeds
speeds_of_interest = [0.1, 10, 20, 30, 40] # 0 is outside the spline knots
pred_df = pd.DataFrame({"speed": speeds_of_interest})
rates = results_speed_spline.predict(pred_df) / BIN_SIZE

print("Spline model — predicted rates:")
for s, r in zip(speeds_of_interest, rates):
    print(f"  speed={s:2f} cm/s → {r:.2f} Hz")

# Peak speed
speed_range = np.linspace(spk_cov_df["speed"].min(), spk_cov_df["speed"].max(), 500)
pred_curve = results_speed_spline.predict(pd.DataFrame({"speed": speed_range})) / BIN_SIZE
peak_speed = speed_range[np.argmax(pred_curve)]
print(f"\nPeak firing at: {peak_speed:.1f} cm/s ({pred_curve.max():.2f} Hz)")
print(f"Rate ratio high/low speed: {pred_curve.max() / pred_curve.min():.2f}x")

Spline model — predicted rates:
  speed=0.100000 cm/s → 14.39 Hz
  speed=10.000000 cm/s → 0.16 Hz
  speed=20.000000 cm/s → 0.36 Hz
  speed=30.000000 cm/s → 0.60 Hz
  speed=40.000000 cm/s → 0.78 Hz

Peak firing at: 0.0 cm/s (15.26 Hz)
Rate ratio high/low speed: 97.93x


In [132]:
fig, ax = plt.subplots(figsize=(7, 4))                                                                  
                                                                                                        
# Observed:                              
speed_bins = np.linspace(spk_cov_df["speed"].min(), spk_cov_df["speed"].max(), 30)
bin_idx = np.digitize(spk_cov_df["speed"], speed_bins) - 1

obs_speed, obs_rate, obs_ci = [], [], []
for b in range(len(speed_bins) - 1):
    sel = bin_idx == b
    if sel.sum() > 50:
        counts = spk_cov_df.loc[sel, "spike_count"].values
        n = len(counts)
        mean = counts.mean() / BIN_SIZE
        sem  = stats.sem(counts) / BIN_SIZE          # SEM in Hz
        obs_speed.append(speed_bins[b:b+2].mean())
        obs_rate.append(mean)
        obs_ci.append(1.96 * sem)                    # 95% CI half-width

obs_speed, obs_rate, obs_ci = map(np.array, [obs_speed, obs_rate, obs_ci])

ax.errorbar(obs_speed, obs_rate, yerr=obs_ci,
            fmt="o", ms=4, color="grey", ecolor="lightgrey",
            elinewidth=1.5, capsize=3, label="observed ± 95% CI", zorder=3)

# Predicted
speed_range = np.linspace(spk_cov_df["speed"].min(), spk_cov_df["speed"].max(), 300)
pred_df = pd.DataFrame({"speed": speed_range})
pred_rate = results_speed_spline.predict(pred_df) / BIN_SIZE

ax.plot(speed_range, pred_rate, color="steelblue", lw=2, label=f"spline fit (df={cr_df})")

ax.set_xlabel("speed (cm/s)")
ax.set_ylabel("firing rate (Hz)")
ax.set_title(f"unit {unit_idx} — speed tuning")
ax.legend()
sns.despine()
plt.tight_layout()


#### Fit on all units

In [ ]:
# speed_spline_model_all = fit_glm_all_units("spike_count ~ cr(speed, df = 4)", cov_df_speed, spike_counts_speed, unit_ids) 

/home/labuser/miniforge3/envs/spyglass/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:445: RuntimeWarning: invalid value encountered in divide
  endog_mu = self._clean(endog / mu)


In [142]:
# speed_spline_model_all["model"] = "speed spline"
# speed_spline_model_all.to_csv(f"{base_dir}/analysis/speed_spline_model_all.csv")
speed_spline_model_all = pd.read_csv(f"{base_dir}/analysis/speed_spline_model_all.csv", index_col=0)
speed_spline_model_all.head(1)

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,111026.932535711,-55509.466267856,89594.658422934,5.000000000,1132036.000000000,False,"{'Intercept': 24265691950.00032, 'cr(speed, df...","{'Intercept': 928919175618.3048, 'cr(speed, df...",100002.515597235,3.000000000,NaN,speed spline


#### Linear position (spline)

In [147]:
pos_mask = cov_df["linear_position"].notna()
cov_df_pos = cov_df[pos_mask].copy()
spike_counts_speed = spike_counts_masked[:, pos_mask]

model_pos_spline = smf.glm("spike_count ~ cr(linear_position, df = 8)",
                           data = spk_cov_df[spk_cov_df["linear_position"].notna()],
                           family = sm.families.Poisson())  

results_pos_spline = model_pos_spline.fit()
print(results_pos_spline.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1171020
Model:                            GLM   Df Residuals:                  1171012
Model Family:                 Poisson   Df Model:                            7
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -14184.
Date:                Wed, 11 Mar 2026   Deviance:                       24413.
Time:                        15:55:18   Pearson chi2:                 1.19e+06
No. Iterations:                   100   Pseudo R-squ. (CS):          0.0007174
Covariance Type:            nonrobust                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

In [148]:
params = results_pos_spline.params
print(f"n_params: {len(params)}")  

pos_min = spk_cov_df["linear_position"].dropna().min()
pos_max = spk_cov_df["linear_position"].dropna().max()
pos_range = np.linspace(pos_min, pos_max, 500)
pred_df = pd.DataFrame({"linear_position": pos_range})
pred_rate = results_pos_spline.predict(pred_df) / BIN_SIZE

peak_pos  = pos_range[np.argmax(pred_rate)]
peak_rate = pred_rate.max()
print(f"Peak firing position: {peak_pos:.1f} cm, rate: {peak_rate:.2f} Hz")

n_params: 9
Peak firing position: 387.9 cm, rate: 2.20 Hz


In [162]:
fig, ax = plt.subplots()

pos_bins = np.linspace(pos_min, pos_max, 40)
bin_idx  = np.digitize(spk_cov_df["linear_position"].dropna(), pos_bins) - 1
pos_df   = spk_cov_df[spk_cov_df["linear_position"].notna()].copy()

obs_pos, obs_rate, obs_ci = [], [], []
for b in range(len(pos_bins) - 1):
    sel = bin_idx == b
    if sel.sum() > 50:
        counts = pos_df.iloc[np.where(sel)[0]]["spike_count"].values
        obs_pos.append(pos_bins[b:b+2].mean())
        obs_rate.append(counts.mean() / BIN_SIZE)
        obs_ci.append(1.96 * stats.sem(counts) / BIN_SIZE)

ax.errorbar(obs_pos, obs_rate, yerr=obs_ci,
            fmt="o", ms=4, color="grey", ecolor="lightgrey",
            elinewidth=1.5, capsize=3, label="observed ± 95% CI", zorder=3)

ax.plot(pos_range, pred_rate, color="steelblue", lw=2, label="spline fit (df=8)")
ax.axvline(peak_pos, color="red", lw=1, linestyle="--", label=f"peak @ {peak_pos:.0f} cm")

ax.set_xlabel("linear position (cm)")
ax.set_ylabel("firing rate (Hz)")
ax.set_title(f"unit {unit_idx} — position tuning")
ax.legend()
sns.despine()
plt.tight_layout()


In [ ]:
graph = sgpl.TrackGraph & {"track_graph_name": "Wtrack_wilbur20210512"}

proj = trialized_position[["linear_position", "projected_x_position", "projected_y_position"]].dropna().sort_values("linear_position")                                                            
                                                                                                        
pos_min = spk_cov_df["linear_position"].dropna().min()
pos_max = spk_cov_df["linear_position"].dropna().max()                                                  
pos_range = np.linspace(pos_min, pos_max, 500)

predicted_rate_hz = results_pos_spline.predict(
    pd.DataFrame({"linear_position": pos_range})) / BIN_SIZE

nearest_idx = np.searchsorted(proj["linear_position"].values, pos_range).clip(0, len(proj) - 1)
pred_x = proj["projected_x_position"].values[nearest_idx]
pred_y = proj["projected_y_position"].values[nearest_idx]

fig, (ax_curve, ax_track) = plt.subplots(1, 2, figsize=(14, 5))

ax_curve.plot(pos_range, predicted_rate_hz, color="steelblue", lw=2)
ax_curve.errorbar(obs_pos, obs_rate, yerr=obs_ci,
            fmt="o", ms=4, color="grey", ecolor="lightgrey",
            elinewidth=1.5, capsize=3, label="observed ± 95% CI", zorder=3)
ax_curve.set_xlabel("linear position (cm)")
ax_curve.set_ylabel("firing rate (Hz)")
ax_curve.set_title(f"unit {unit_idx} — position tuning")

graph.plot_track_graph(ax=ax_track, draw_edge_labels=False)
for ln in ax_track.lines:
    ln.set_color("lightgrey")

sc = ax_track.scatter(pred_x, pred_y, c=predicted_rate_hz,
                    cmap="hot_r", s=8, zorder=3,
                    vmin=predicted_rate_hz.min(), vmax=predicted_rate_hz.max())
plt.colorbar(sc, ax=ax_track, label="firing rate (Hz)")
ax_track.set_xlabel("x position (cm)")
ax_track.set_ylabel("y position (cm)")
ax_track.set_title(f"unit {unit_idx} — rate on track")

sns.despine()
plt.tight_layout()
